# Part 4 — The Work IQ tool catalog

Parts 1–3 introduced Work IQ one idea at a time. This notebook is the **complete tool catalog** —
all **10 tools** the Work IQ MCP server exposes, in the three official categories.

> **Fewer tools, more paths.** Every tool operates on a **relative resource path** (a Microsoft
> Graph endpoint). New workloads add *paths*, not tools — the surface stays fixed at 10.

| Category | Tools | What they do |
| --- | --- | --- |
| **Copilot** | `ask`, `list_agents` | Natural-language reasoning over your work context; discover agents |
| **Schema** | `search_paths`, `get_schema` | Discover which paths exist and their exact request shape at runtime |
| **Entity** | `fetch`, `call_function`, `create_entity`, `update_entity`, `delete_entity`, `do_action` | CRUD + actions on Microsoft 365 resources |

Everything runs **as the signed-in user**, honoring their Microsoft 365 permissions and sensitivity
labels. The write tools mutate real data — always confirm with a human before they run.

Docs: [Work IQ MCP tool reference](https://learn.microsoft.com/microsoft-365/copilot/extensibility/work-iq/mcp/tool-reference)

In [19]:
import sys, os, json
sys.path.append(os.path.abspath(".."))
from notebooks._shared import get_user_token, call_mcp, call_tool

token = get_user_token()
def mcp_call(name, arguments=None):
    return call_tool(token, name, arguments or {})
print("Signed in - Work IQ tools ready. Token:", token[:16], "...")

Signed in - Work IQ tools ready. Token: eyJ0eXAiOiJKV1Qi ...


## The complete catalog, straight from the server

`tools/list` is authoritative: it returns every tool with its input schema. Print each tool's name
and required arguments — the ground truth for everything below.

In [20]:
tools = call_mcp(token, "tools/list", {})["result"]["tools"]
print(f"{len(tools)} tools:\n")
for t in tools:
    req = t.get("inputSchema", {}).get("required", [])
    print(f"{t['name']:<15} required={req}")

10 tools:

ask             required=['question']
list_agents     required=[]
delete_entity   required=['entityUrl']
do_action       required=['actionUrl', 'jsonBody']
create_entity   required=['parentUrl', 'jsonBody']
get_schema      required=[]
search_paths    required=[]
call_function   required=['functionUrl']
fetch           required=['entityUrls']
update_entity   required=['entityUrl', 'jsonBody']


## Copilot tools — `ask`, `list_agents`

`ask` is the highest-leverage tool: pose a question in plain language and Work IQ (Microsoft 365
Copilot) plans retrieval across mail, chats, meetings, and files, then returns a synthesized answer
with citations. Start here — reach for the lower-level tools only when you need precise control.

In [21]:
resp = mcp_call("ask", {"question":
    "Summarize my most recent emails and Teams chats, and list anything that needs my attention."})
print(resp["result"]["structuredContent"]["answer"])

I found **recent email and Teams activity** in your Microsoft 365 data. The search returned several recent email messages and Teams conversations. [1](https://outlook.office365.com/owa/?ItemID=AAMkAGU5YWI5ZGMyLWU2MDQtNDIzMi05YzE0LTE2YWY2NmEwYTFiMwBGAAAAAACK73wHO6G9SbOuNTn1ktl5BwApssw8XhI5R4eaNP1xXhr5AAAAAAEMAAApssw8XhI5R4eaNP1xXhr5AABBSwksAAA%3d&exvsurl=1&viewmodel=ReadMessageItem#4bf8c2)[2](https://teams.microsoft.com/l/message/19:09d2d679-fdea-42a9-aa1e-7eaacb609ea3_47c55e51-f96d-47fc-b396-d55bb88b7c6b@unq.gbl.spaces/1785211492461?context=%7B%22contextType%22:%22chat%22%7D#7e71af)

## Needs Your Attention

### 1. Azure Copilot access settings review (highest priority)
You received [Action recommended: Review Azure Copilot agent access settings before 1 August 2026](https://outlook.office365.com/owa/?ItemID=AAMkAGU5YWI5ZGMyLWU2MDQtNDIzMi05YzE0LTE2YWY2NmEwYTFiMwBGAAAAAACK73wHO6G9SbOuNTn1ktl5BwApssw8XhI5R4eaNP1xXhr5AAAAAAEMAAApssw8XhI5R4eaNP1xXhr5AABBSwksAAA%3d&exvsurl=1&viewmodel=ReadM

`list_agents` lists the agents you can route `ask` to (via its optional `agentId`) — the built-in
Microsoft 365 Copilot agent is always included.

In [22]:
res = mcp_call("list_agents", {})["result"]
agents = res.get("structuredContent") or json.loads(res["content"][0]["text"])
print(json.dumps(agents, indent=2)[:600])

[
  {
    "agentId": "P_b2f25cb5-1ccd-b7ed-8a74-35d2b43e0130.declarativeAgent",
    "name": "Planner Agent",
    "provider": "Microsoft Corporation"
  },
  {
    "agentId": "P_c0fd1ab0-cbf3-7eb9-1a7d-2d823549ef31.8ad61c39-5b6e-447c-b26a-a64eee436502",
    "name": "Microsoft 365 Admin",
    "provider": "Microsoft Corporation"
  },
  {
    "agentId": "P_a64931a6-8080-b1e7-e057-8e775a24ed66.declarativeAgent",
    "name": "Surveys",
    "provider": "Microsoft Corporation"
  },
  {
    "agentId": "T_923982d9-73cc-1316-1cc0-adbc280d27b8.144e33e9-f685-4ae8-bcbe-cb05d0ca59ec",
    "name": "workmate-ag


## Schema tools — `search_paths`, `get_schema`

Before you read or write, **discover** what's reachable. `search_paths` lists the Graph paths (and
the operations each supports) matching a regex `filter`.

In [23]:
# search_paths returns structuredContent.paths: a list of {path, operations}.
resp = mcp_call("search_paths", {"filter": "messages|events"})
for p in resp["result"]["structuredContent"]["paths"][:10]:
    print(f"{p['path']:<48} -> {', '.join(p['operations'])}")

/chats/{chat-id}/messages                        -> fetch, create
/chats/{chat-id}/messages/{chatMessage-id}       -> fetch, update, delete
/chats/{chat-id}/messages/{chatMessage-id}/hostedContents -> fetch
/chats/{chat-id}/messages/{chatMessage-id}/hostedContents/{chatMessageHostedContent-id} -> fetch, update, delete
/chats/{chat-id}/messages/delta()                -> fetch
/chats/{chat-id}/pinnedMessages                  -> fetch, create
/chats/{chat-id}/pinnedMessages/{pinnedChatMessageInfo-id} -> fetch, update, delete
/chats/getAllMessages()                          -> fetch
/chats/getAllRetainedMessages()                  -> fetch
/groups/{group-id}/calendar/events/{event-id}    -> fetch, update, delete


`get_schema` returns the request shape for a path. Pass `operationType` — one of
`fetch | create | update` — to get the shape for what you're about to do.

In [24]:
read_schema = mcp_call("get_schema", {"path": "/me/messages", "operationType": "fetch"})
print("FETCH shape for /me/messages:\n", read_schema["result"]["content"][0]["text"][:450])

create_schema = mcp_call("get_schema", {"path": "/me/events", "operationType": "create"})
print("\nCREATE shape for /me/events:\n", create_schema["result"]["content"][0]["text"][:450])

FETCH shape for /me/messages:
 ; CDDL definition for microsoft.graph.messageCollectionResponse

microsoft-graph-messageCollectionResponse = {
  BaseCollectionPaginationCountResponse,
  ? "value": [* microsoft-graph-message]
}

BaseCollectionPaginationCountResponse = {
  ? "@odata.count": (int / null),
  ? "@odata.nextLink": (tstr / null)
}

microsoft-graph-message = {
  microsoft-graph-outlookItem,
  ? "bccRecipients": [* microsoft-graph-recipient],
  ? "body": microsoft

CREATE shape for /me/events:
 ; CDDL definition for microsoft.graph.event

microsoft-graph-event = {
  microsoft-graph-outlookItem,
  ? "allowNewTimeProposals": (bool / null),
  ? "attendees": [* microsoft-graph-attendee],
  ? "body": microsoft-graph-itemBody / { * tstr => any },
  ? "bodyPreview": (tstr / null),
  ? "cancelledOccurrences": [* tstr],
  ? "end": microsoft-graph-dateTimeTimeZone / { * tstr => any },
  ? "hasAttachments": (bool / null),
  ? "hideAttendees": (b


## Entity tools — read: `fetch`, `call_function`

`fetch` reads one or more entities by URL (`entityUrls`). Use it when you know exactly what you want,
e.g. the last five messages with just the fields you care about.

In [25]:
msgs = mcp_call("fetch", {"entityUrls": ["/me/messages?$top=5&$select=subject,from,receivedDateTime"]})
for res in msgs["result"]["structuredContent"]["results"]:
    for m in res["data"].get("value", [])[:5]:
        print(m.get("receivedDateTime"), "-", m.get("subject"))

2026-07-23T14:24:46Z - Action recommended: Review Azure Copilot agent access settings before 1 August 2026
2026-07-23T05:58:35Z - Work Mate is trying to reach you in Microsoft Teams
2026-07-21T16:09:28Z - AI Instance assigned in Microsoft 365
2026-07-19T13:12:08Z - Your weekly PIM digest for CA M365 2609 (ID: a9d9510e-7131-4355-8b7e-37e7b1e99862)
2026-07-12T00:53:54Z - Your weekly PIM digest for CA M365 2609 (ID: a9d9510e-7131-4355-8b7e-37e7b1e99862)


`call_function` invokes a read-only Microsoft Graph **function** that computes derived data —
schedules, deltas, search results — via `functionUrl`. Here: this week's calendar view.

In [26]:
from datetime import datetime, timedelta, timezone
# Graph calendarView wants URL-safe ISO timestamps: use a Z suffix, not
# isoformat()'s +00:00 (the + becomes a space in the query string -> 400).
start = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
end = (datetime.now(timezone.utc) + timedelta(days=7)).strftime("%Y-%m-%dT%H:%M:%SZ")
resp = mcp_call("call_function",
    {"functionUrl": f"/me/calendarView?startdatetime={start}&enddatetime={end}"})
for e in (resp["result"]["structuredContent"]["data"] or {}).get("value", [])[:5]:
    print(e["start"]["dateTime"], "-", e.get("subject"))

2026-07-29T09:00:00.0000000 - Weekly team sync
2026-07-30T14:00:00.0000000 - Professional Claw Hammer launch review
2026-08-01T15:00:00.0000000 - Q3 budget planning
2026-08-02T11:00:00.0000000 - 1:1 with mentor


## Entity tools — write: `create_entity`, `update_entity`, `delete_entity`, `do_action`

These four **mutate real data as the signed-in user**. Two gotchas from the docs:

- `create_entity` takes **`parentUrl`** (the collection); `update_entity` / `delete_entity` take
  **`entityUrl`** (the specific item); `do_action` takes **`actionUrl`**.
- `jsonBody` must be a **JSON-encoded string**, not a JSON object — so we `json.dumps(...)` it.

Get every payload shape from `get_schema`, and **always confirm with a human first**.

In [27]:
# All dry-run: flip WRITE=True only after a human approves the payload.
WRITE = False

create_args = {"parentUrl": "/me/events", "jsonBody": json.dumps({
    "subject": "Q3 budget sync",
    "start": {"dateTime": "2026-08-01T15:00:00", "timeZone": "UTC"},
    "end":   {"dateTime": "2026-08-01T15:30:00", "timeZone": "UTC"},
})}
update_args = {"entityUrl": "/me/events/<event-id>",
               "jsonBody": json.dumps({"subject": "Q3 budget sync (moved)"})}
delete_args = {"entityUrl": "/me/events/<event-id>"}

for tool, args in [("create_entity", create_args),
                   ("update_entity", update_args),
                   ("delete_entity", delete_args)]:
    if WRITE:
        print(tool, "->", mcp_call(tool, args))
    else:
        print(f"[dry run] {tool}({json.dumps(args)})")

[dry run] create_entity({"parentUrl": "/me/events", "jsonBody": "{\"subject\": \"Q3 budget sync\", \"start\": {\"dateTime\": \"2026-08-01T15:00:00\", \"timeZone\": \"UTC\"}, \"end\": {\"dateTime\": \"2026-08-01T15:30:00\", \"timeZone\": \"UTC\"}}"})
[dry run] update_entity({"entityUrl": "/me/events/<event-id>", "jsonBody": "{\"subject\": \"Q3 budget sync (moved)\"}"})
[dry run] delete_entity({"entityUrl": "/me/events/<event-id>"})


`do_action` is the action verb — send mail, reply, accept an event. Here, sending mail:
`actionUrl` + a JSON-encoded `jsonBody`.

In [28]:
SEND = False  # flip to True to actually send
action_url = "/me/sendMail"
json_body = json.dumps({
    "Message": {
        "subject": "Re: your note",
        "body": {"contentType": "Text", "content": "<approved draft here>"},
        "toRecipients": [{"emailAddress": {"address": "manager@contoso.com"}}],
    },
    "SaveToSentItems": True,
})
if SEND:
    print(mcp_call("do_action", {"actionUrl": action_url, "jsonBody": json_body}))
else:
    print("Dry run - set SEND=True to send. do_action args:")
    print(json.dumps({"actionUrl": action_url, "jsonBody": json_body}, indent=2))

Dry run - set SEND=True to send. do_action args:
{
  "actionUrl": "/me/sendMail",
  "jsonBody": "{\"Message\": {\"subject\": \"Re: your note\", \"body\": {\"contentType\": \"Text\", \"content\": \"<approved draft here>\"}, \"toRecipients\": [{\"emailAddress\": {\"address\": \"manager@contoso.com\"}}]}, \"SaveToSentItems\": true}"
}


### Recap

- **Copilot:** `ask` is the front door — one question, retrieval planned across your work context;
  `list_agents` shows who you can route to.
- **Schema:** `search_paths` → `get_schema` discover reachable paths and their exact shapes.
- **Entity (read):** `fetch` (entities by URL) and `call_function` (computed Graph functions).
- **Entity (write):** `create_entity` / `update_entity` / `delete_entity` and `do_action` — real
  mutations, `jsonBody` as a JSON-encoded string, human-in-the-loop.
- The pattern that scales: **ask / read context → draft → confirm → write**.

**Next:** wrap this toolset into an agent — see the Foundry IQ + Work IQ notebook and `src/workmate-autopilot/`.